# Derivative_Pricing_Group_Work_Project_1_M5_15283 

## Import libraries we will need

In [69]:
import numpy as np

## Question 5 

In [70]:
# Define functions for binomial tree option pricing (European call and put)
def binomial_call_full(S_ini, K, T, r, u, d, N):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # Risk neutral probs
    C = np.zeros([N + 1, N + 1])  # Call prices
    S = np.zeros([N + 1, N + 1])  # Underlying price
    for i in range(0, N + 1):
        C[N, i] = max(S_ini * (u ** (i)) * (d ** (N - i)) - K, 0)
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i])
            S[j, i] = S_ini * (u ** (i)) * (d ** (j - i))
    return C[0, 0], C, S


def binomial_put_full(S_ini, K, T, r, u, d, N):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # Risk neutral probs
    P = np.zeros([N + 1, N + 1])  # Call prices
    S = np.zeros([N + 1, N + 1])  # Underlying price
    for i in range(0, N + 1):
        P[N, i] = max(K - (S_ini * (u ** (i)) * (d ** (N - i))), 0)
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            P[j, i] = np.exp(-r * dt) * (p * P[j + 1, i + 1] + (1 - p) * P[j + 1, i])
            S[j, i] = S_ini * (u ** (i)) * (d ** (j - i))
    return P[0, 0], P, S

In [71]:
S_ini = 100
K = 100
r = 0.05
sigma = 0.20
T = 0.25
N = 3
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = np.exp(-sigma * np.sqrt(dt))

call_price_european_q5, C_european_q5, S_european_q5 = binomial_call_full(S_ini, K, T, r, u, d, N)
print("Price at t=0 for call option is $", "{:.4f}".format(call_price_european_q5))

Price at t=0 for call option is $ 4.9443


In [72]:
put_price_european_q5, P_european_q5, S_european_q5 = binomial_put_full(S_ini, K, T, r, u, d, N)
print("Price at t=0 for put option is $", "{:.4f}".format(put_price_european_q5))

Price at t=0 for put option is $ 3.7021


## Question 8

In [73]:
# Define functions for binomial tree option pricing (American call and put)
# opttype is either "C" for call or "P" for put
def american_option(S_ini, K, T, r, u, d, N, opttype):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # call prices
    S = np.zeros([N + 1, N + 1])  # underlying price

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (
                p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i]
            )  # Computing the European option prices
            S[j, i] = (
                S_ini * (u ** (i)) * (d ** (j - i))
            )  # Underlying evolution for each node
            if opttype == "C":
                C[j, i] = max(
                    C[j, i], S[j, i] - K
                )  # Decision between the European option price and the payoff from early-exercise
            else:
                C[j, i] = max(
                    C[j, i], K - S[j, i]
                )  # Decision between the European option price and the payoff from early-exercise

    return C[0, 0], C, S

In [74]:
call_price_american_q8, C_american_q8, S_american_q8 = american_option(S_ini, K, T, r, u, d, N, "C")
print("Price at t=0 for American call option is $", "{:.4f}".format(call_price_american_q8))

Price at t=0 for American call option is $ 4.9443


In [75]:
put_price_american_q8, P_american_q8, S_american_q8 = american_option(S_ini, K, T, r, u, d, N, "P")
print("Price at t=0 for American put option is $", "{:.4f}".format(put_price_american_q8))

Price at t=0 for American put option is $ 3.7964


## Question 11

In [76]:
# show that the European call and put satisfy the put-call parity
# we use the values calculated in question 5 for the European call and put
round(call_price_european_q5 + K*np.exp(-r*T), 4) == round(put_price_european_q5 + S_ini, 4)

np.True_

## Question 12

In [77]:
# show that the American call and put do not satisfy the put-call parity
# we use the values calculated in question 8 for the American call and put
round(call_price_american_q8 + K*np.exp(-r*T), 4) == round(put_price_american_q8 + S_ini, 4)

np.False_

## Question 13

In [78]:
# Confirm that the American call is more expensive than the European call
print("Is the European call price less than or equal to the American call price?", call_price_european_q5 <= call_price_american_q8)
print("Price difference between American and European call option is $", "{:.4f}".format(call_price_american_q8 - call_price_european_q5))

Is the European call price less than or equal to the American call price? True
Price difference between American and European call option is $ 0.0000


## Question 14

In [79]:

# Confirm that the American put is more expensive than the European put
print("Is the European put price less than or equal to the American put price?", put_price_european_q5 <= put_price_american_q8)
print("Price difference between American and European put option is $", "{:.4f}".format(put_price_american_q8 - put_price_european_q5))

Is the European put price less than or equal to the American put price? True
Price difference between American and European put option is $ 0.0943
